# 06 - Dropping Missing Values (Hands-On)

Every line from `06_Dropping_Missing_Values.MD`, run for real, broken into small enough pieces that each one's own output is visible - no step is left as "trust me."

Dataset: `loans.csv`, same one used in `05_Data_Cleaning_Practice.ipynb`.

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("loans.csv")
dataset.shape

(618, 13)

## Method 1: Drop the Column (>= 50% missing)

Building `missing_pct = (dataset.isnull().sum() / dataset.shape[0]) * 100` one piece at a time.

In [8]:
dataset.isnull().head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False


`.isnull()` turns every cell into `True`/`False`. `.head()` just previews the first 5 rows instead of all 618.

In [9]:
dataset.isnull().sum()

Loan_ID               0
Gender               13
Married               6
Dependents           15
Education             9
Self_Employed        32
ApplicantIncome       2
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     21
Credit_History       50
Property_Area         9
Loan_Status           0
dtype: int64

`.sum()` adds up the `True`s **down each column** (default `axis=0`) - one count per column.

In [10]:
dataset.shape

(618, 13)

In [11]:
dataset.shape[0]

618

`.shape` is `(rows, columns)`; `[0]` picks out just the row count.

In [12]:
dataset.isnull().sum() / dataset.shape[0]

Loan_ID              0.000000
Gender               0.021036
Married              0.009709
Dependents           0.024272
Education            0.014563
Self_Employed        0.051780
ApplicantIncome      0.003236
CoapplicantIncome    0.000000
LoanAmount           0.035599
Loan_Amount_Term     0.033981
Credit_History       0.080906
Property_Area        0.014563
Loan_Status          0.000000
dtype: float64

Each column's missing count divided by total rows = the fraction missing.

In [13]:
missing_pct = (dataset.isnull().sum() / dataset.shape[0]) * 100
missing_pct

Loan_ID              0.000000
Gender               2.103560
Married              0.970874
Dependents           2.427184
Education            1.456311
Self_Employed        5.177994
ApplicantIncome      0.323625
CoapplicantIncome    0.000000
LoanAmount           3.559871
Loan_Amount_Term     3.398058
Credit_History       8.090615
Property_Area        1.456311
Loan_Status          0.000000
dtype: float64

`* 100` turns the fraction into a percentage - this is the full Line 1 from the `.MD` file, now built from its pieces.

In [14]:
missing_pct >= 50

Loan_ID              False
Gender               False
Married              False
Dependents           False
Education            False
Self_Employed        False
ApplicantIncome      False
CoapplicantIncome    False
LoanAmount           False
Loan_Amount_Term     False
Credit_History       False
Property_Area        False
Loan_Status          False
dtype: bool

Comparing every value to `50` produces a same-shaped Series of `True`/`False`. All `False` here - no column is close.

In [10]:
missing_pct[missing_pct >= 50]

Series([], dtype: float64)

**Boolean indexing**: keep only the entries where the mask is `True`. Empty, because nothing qualifies.

In [15]:
cols_to_drop = missing_pct[missing_pct >= 50].index
cols_to_drop

Index([], dtype='str')

`.index` pulls out just the column *names* from what's left - an empty list of names.

In [16]:
dataset_cleaned = dataset.drop(columns=cols_to_drop)
dataset_cleaned.shape

(618, 13)

Dropping an empty list of columns removes nothing - shape is unchanged. That's the correct answer, not a bug.

### The two idiomatic one-liners

Same outcome, written the way you'll see it in other people's code.

In [13]:
len(dataset)

618

In [14]:
0.5 * len(dataset)

309.0

In [15]:
int(0.5 * len(dataset))

309

`len()` on a DataFrame gives the row count. `thresh` needs a whole number, hence `int(...)`.

In [16]:
dataset.dropna(axis=1, thresh=int(0.5 * len(dataset))).shape

(618, 13)

`axis=1` here means "evaluate **columns**" (not the fill-direction meaning from `07_Filling_Missing_Values.MD`). `thresh=309` keeps a column only if it has at least 309 non-null values. Same `(618, 13)` result.

In [17]:
dataset.isnull().mean()

Loan_ID              0.000000
Gender               0.021036
Married              0.009709
Dependents           0.024272
Education            0.014563
Self_Employed        0.051780
ApplicantIncome      0.003236
CoapplicantIncome    0.000000
LoanAmount           0.035599
Loan_Amount_Term     0.033981
Credit_History       0.080906
Property_Area        0.014563
Loan_Status          0.000000
dtype: float64

`.mean()` of `True`/`False` *is* the proportion `True` - same numbers as `missing_pct`, just as fractions instead of percentages.

In [18]:
dataset.isnull().mean() < 0.5

Loan_ID              True
Gender               True
Married              True
Dependents           True
Education            True
Self_Employed        True
ApplicantIncome      True
CoapplicantIncome    True
LoanAmount           True
Loan_Amount_Term     True
Credit_History       True
Property_Area        True
Loan_Status          True
dtype: bool

In [19]:
dataset.loc[:, dataset.isnull().mean() < 0.5].shape

(618, 13)

`.loc[:, mask]` - `:` means *every row*, the boolean Series picks which *columns* survive. A third way to write the exact same idea.

## Method 2: Drop the Row

For scattered, low-volume missingness once the worst columns are already ruled out.

In [20]:
dataset.dropna().shape

(457, 13)

In [21]:
dataset.shape[0] - dataset.dropna().shape[0]

161

Default `dropna()` drops a row if **any** of its 13 columns is missing - **161 rows gone**, even though no single column was worse than ~8%.

In [22]:
dataset.dropna(subset=["Loan_Status"]).shape

(618, 13)

`subset=` only checks the named column(s). `Loan_Status` has 0 missing, so **0 rows dropped**.

In [23]:
dataset.notnull().sum(axis=1).head(10)

0    13
1    13
2    13
3    13
4    13
5    12
6    13
7    12
8    13
9    13
dtype: int64

`.sum(axis=1)` counts non-null values **per row** this time - one number per row instead of per column.

In [24]:
dataset.notnull().sum(axis=1).value_counts().sort_index()

10      1
11     16
12    144
13    457
Name: count, dtype: int64

The full distribution: 1 row has only 10 non-null values (3 missing), 16 rows have 11 (2 missing), 144 have 12 (1 missing), and 457 rows are fully complete.

In [25]:
dataset.dropna(thresh=12).shape

(601, 13)

In [26]:
dataset.shape[0] - dataset.dropna(thresh=12).shape[0]

17

`thresh=12` keeps rows with 12 or 13 non-null values (`144 + 457 = 601`) and drops the rest (`1 + 16 = 17`) - matches the distribution above exactly.

## The Real Numbers - proving the 161-row drop

In [27]:
dataset.isnull().any(axis=1).head(10)

0    False
1    False
2    False
3    False
4    False
5     True
6    False
7     True
8    False
9    False
dtype: bool

`.any(axis=1)` - for **each row**, `True` if at least one of its columns is missing.

In [28]:
dataset.isnull().any(axis=1).sum()

np.int64(161)

Summed up: **161 rows** have a gap *somewhere*, even though every individual column's own miss-rate is under 9%. That's a union across 13 columns, not any single column's rate - the actual mechanism behind Method 2 removing more rows than intuition suggests.

## Recap

- **Method 1** (column drop, >= 50%) removes **0 columns** from `loans.csv` - nothing here is bad enough to qualify.
- **Method 2** (row drop) removes **161 rows** with a blind `dropna()`, **0 rows** with a targeted `subset=["Loan_Status"]`, and **17 rows** with a `thresh=12` tolerance.
- Full prose explanation of *why* each of these numbers comes out this way: `06_Dropping_Missing_Values.MD`.